![JohnSnowLabs](https://nlp.johnsnowlabs.com/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/visual-nlp-workshop/blob/master/tutorials/Certification_Trainings/05.00.Spark_OCR_Streaming_PDF.ipynb)

If you are using the `johnsnowlabs` library, please use this [05.00.Spark_OCR_Streaming_PDF](https://github.com/JohnSnowLabs/visual-nlp-workshop/blob/master/tutorials/Certification_Trainings_JSL/05.00.Spark_OCR_Streaming_PDF.ipynb) notebook.

## Spark OCR Streaming

## Blogposts and videos

- [Text Detection in Spark OCR](https://medium.com/spark-nlp/text-detection-in-spark-ocr-dcd8002bdc97)

- [Table Detection & Extraction in Spark OCR](https://medium.com/spark-nlp/table-detection-extraction-in-spark-ocr-50765c6cedc9)

- [Extract Tabular Data from PDF in Spark OCR](https://medium.com/spark-nlp/extract-tabular-data-from-pdf-in-spark-ocr-b02136bc0fcb)

- [Signature Detection in Spark OCR](https://medium.com/spark-nlp/signature-detection-in-spark-ocr-32f9e6f91e3c)

- [GPU image pre-processing in Spark OCR](https://medium.com/spark-nlp/gpu-image-pre-processing-in-spark-ocr-3-1-0-6fc27560a9bb)

- [How to Setup Spark OCR on UBUNTU - Video](https://www.youtube.com/watch?v=cmt4WIcL0nI)


**More examples here**

https://github.com/JohnSnowLabs/spark-ocr-workshop

In [ ]:
# NBVAL_SKIP
import json, os
import sys

if 'google.colab' in sys.modules:
    from google.colab import files

    if 'spark_ocr.json' not in os.listdir():
      license_keys = files.upload()
      os.rename(list(license_keys.keys())[0], 'spark_ocr.json')

with open('spark_ocr.json') as f:
    license_keys = json.load(f)

# Defining license key-value pairs as local variables
locals().update(license_keys)

In [ ]:
# NBVAL_SKIP
# Installing pyspark and spark-nlp
%pip install --upgrade -q pyspark==3.4.1 spark-nlp==$PUBLIC_VERSION

# Installing Spark OCR
#! pip uninstall spark-ocr -Y
%pip install spark-ocr==$OCR_VERSION --extra-index-url=https://pypi.johnsnowlabs.com/$SPARK_OCR_SECRET --upgrade

<b><h1><font color='darkred'>!!! ATTENTION !!! </font><h1><b>

<b>Running the next cell will <font color='darkred'>automatically restart the Colab runtime</font>. This is expected — once it restarts, just continue running the cells below.<b>

In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
# NBVAL_SKIP
import json, os

with open("spark_ocr.json", 'r') as f:
  license_keys = json.load(f)

# Adding license key-value pairs to environment variables
os.environ.update(license_keys)

# Defining license key-value pairs as local variables
locals().update(license_keys)

In [2]:
import pyspark
import sparkocr
import json
import os

from pyspark.sql import SparkSession
from pyspark.ml import PipelineModel
import pyspark.sql.functions as f

from sparkocr import start
from sparkocr.transformers import *
from sparkocr.utils import display_images
from sparkocr.enums import *

## Initialization of spark session

In [3]:
# Start spark
spark = start(secret=SPARK_OCR_SECRET)

Spark version: 3.4.1
Spark NLP version: 6.4.0
Spark OCR version: 6.4.0



In [4]:
from pyspark.ml import PipelineModel
from pyspark.sql.functions import *

from sparkocr.transformers import *

In [5]:
# Download a sample PDF from the repo
!mkdir -p ./data/pdfs
!wget -q -O ./data/pdfs/noised.pdf https://raw.githubusercontent.com/JohnSnowLabs/visual-nlp-workshop/master/tutorials/Certification_Trainings/data/pdfs/noised.pdf

# fill path to folder with PDF's here
dataset_path = "./data/pdfs/*.pdf"

In [6]:
# read one file for infer schema
pdfs_df = spark.read.format("binaryFile").load(dataset_path).limit(1)

## Define OCR pipeline

In [18]:
# Transform binary to image
pdf_to_image = PdfToImage()
pdf_to_image.setOutputCol("image")

# Run OCR for each region
ocr = ImageToText()
ocr.setInputCol("image")
ocr.setOutputCol("text")
ocr.setConfidenceThreshold(60)

# OCR pipeline
pipeline = PipelineModel(stages=[
    pdf_to_image,
    ocr
])

## Define streaming pipeline and start it
Note: each start erase previous results

In [19]:
# count of files in one microbatch
maxFilesPerTrigger = 4

# read files as stream
pdf_stream_df = spark.readStream \
.format("binaryFile") \
.schema(pdfs_df.schema) \
.option("maxFilesPerTrigger", maxFilesPerTrigger) \
.load(dataset_path)

# process files using OCR pipeline
result = pipeline.transform(pdf_stream_df).withColumn("timestamp", current_timestamp())

# store results to memory table
query = result.writeStream \
 .format('memory') \
 .queryName('result') \
 .start()

In [20]:
import time
time.sleep(10)

# get progress of streamig job
query.lastProgress

{'id': '7205314f-08ac-4ca7-b65a-f530069e68f8',
 'runId': 'dbc659cc-3803-4403-a549-792f2b1ed0cd',
 'name': 'result',
 'timestamp': '2026-07-08T12:23:04.992Z',
 'batchId': 0,
 'numInputRows': 1,
 'inputRowsPerSecond': 0.0,
 'processedRowsPerSecond': 0.18060321473722232,
 'durationMs': {'addBatch': 5243,
  'commitOffsets': 64,
  'getBatch': 11,
  'latestOffset': 74,
  'queryPlanning': 77,
  'triggerExecution': 5537,
  'walCommit': 65},
 'stateOperators': [],
 'sources': [{'description': 'FileStreamSource[file:/content/data/pdfs/*.pdf]',
   'startOffset': None,
   'endOffset': {'logOffset': 0},
   'latestOffset': None,
   'numInputRows': 1,
   'inputRowsPerSecond': 0.0,
   'processedRowsPerSecond': 0.18060321473722232}],
 'sink': {'description': 'MemorySink', 'numOutputRows': 1}}

In [21]:
time.sleep(10)
# need to run for stop steraming job
query.stop()

## Show results from 'result' table

In [22]:
# count of processed records (number of processed pages in results)
spark.table("result").count()

1

In [23]:
# show results
spark.table("result").select("timestamp","pagenum", "path", "text").show(10)

+--------------------+-------+--------------------+--------------------+
|           timestamp|pagenum|                path|                text|
+--------------------+-------+--------------------+--------------------+
|2026-07-08 12:23:...|      0|file:/content/dat...| \n\n \n\n \n\nSa...|
+--------------------+-------+--------------------+--------------------+



## Run streaming job for storing results to disk

In [41]:
query = result.select("text").writeStream \
 .format('text') \
 .option("path", "results/") \
 .option("checkpointLocation", "checkpointDir") \
 .start()

In [42]:
time.sleep(10)
# get progress of streamig job
query.lastProgress

{'id': '512f03f1-7662-4249-9deb-61cd2a29294f',
 'runId': '07166a59-c29b-4429-a429-2a487988964c',
 'name': None,
 'timestamp': '2026-07-08T12:26:59.125Z',
 'batchId': 0,
 'numInputRows': 1,
 'inputRowsPerSecond': 0.0,
 'processedRowsPerSecond': 0.25614754098360654,
 'durationMs': {'addBatch': 3761,
  'commitOffsets': 31,
  'getBatch': 7,
  'latestOffset': 34,
  'queryPlanning': 34,
  'triggerExecution': 3904,
  'walCommit': 36},
 'stateOperators': [],
 'sources': [{'description': 'FileStreamSource[file:/content/data/pdfs/*.pdf]',
   'startOffset': None,
   'endOffset': {'logOffset': 0},
   'latestOffset': None,
   'numInputRows': 1,
   'inputRowsPerSecond': 0.0,
   'processedRowsPerSecond': 0.25614754098360654}],
 'sink': {'description': 'FileSink[results/]', 'numOutputRows': -1}}

In [43]:
time.sleep(10)
# need to run for stop steraming job
query.stop()

## Read results from disk

In [44]:
# NBVAL_SKIP
results = spark.read.format("text").load("results/*.txt")
results.show(50, truncate=False)

+--------------------------------------------------------------------+
|value                                                               |
+--------------------------------------------------------------------+
|                                                                    |
|                                                                    |
|                                                                    |
|                                                                    |
|                                                                    |
|                                                                    |
|Sample specifications written by                                    |
|                                                                    |
|                                                                    |
|                                                                    |
|~ , BLEND CASING RECASING                                           |
|= OLD

In [46]:
results.sample(.1).show(truncate=False)

+---------------------------+
|value                      |
+---------------------------+
|Control for Sample No. 5030|
|                           |
|Filter Production--- -- ,  |
|                           |
|                           |
+---------------------------+



## Clean results and checkpoint folders

In [40]:
%%bash
rm -r -f results
rm -r -f checkpointDir